In [4]:
!git clone https://github.com/bremsstrahlung-57/practicum-project

fatal: destination path 'practicum-project' already exists and is not an empty directory.


In [15]:
!pip install torch_pruning -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 7.6 MB/s eta 0:00:00


In [5]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
import wandb
wandb.login(WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet50, resnet18
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch_pruning as tp
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=transform_test)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                           shuffle=True, num_workers=2)
testloader  = torch.utils.data.DataLoader(testset, batch_size=256,
                                           shuffle=False, num_workers=2)

100%|██████████| 170M/170M [10:55<00:00, 260kB/s]


In [8]:
def get_cifar_resnet50():
    model = resnet50(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(2048, 10)
    return model

teacher = get_cifar_resnet50().to(device)

In [9]:
TEACHER_EPOCHS = 50
teacher_optimizer = torch.optim.SGD(teacher.parameters(), lr=0.1,
                                     momentum=0.9, weight_decay=5e-4)
teacher_scheduler = CosineAnnealingLR(teacher_optimizer, T_max=TEACHER_EPOCHS, eta_min=1e-6)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

wandb.init(
    project="cnn-compression",
    name="teacher-resnet50",
    config={
        "epochs": TEACHER_EPOCHS,
        "lr": 0.1,
        "model": "resnet50-cifar",
        "label_smoothing": 0.1,
    }
)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        correct += model(inputs).argmax(1).eq(targets).sum().item()
        total   += inputs.size(0)
    return 100. * correct / total


In [10]:
best_teacher_acc = 0.0
for epoch in range(1, TEACHER_EPOCHS + 1):
    teacher.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, targets in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)
        teacher_optimizer.zero_grad()
        outputs = teacher(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        teacher_optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        correct    += outputs.argmax(1).eq(targets).sum().item()
        total      += inputs.size(0)
    teacher_scheduler.step()

    val_acc = evaluate(teacher, testloader)
    train_acc = 100. * correct / total
    wandb.log({"epoch": epoch, "train/acc": train_acc, "val/acc": val_acc,
               "lr": teacher_scheduler.get_last_lr()[0]})
    print(f"[Teacher] Epoch {epoch:02d}/{TEACHER_EPOCHS} | "
          f"Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")

    if val_acc > best_teacher_acc:
        best_teacher_acc = val_acc
        torch.save(teacher, '/content/practicum-project/models/teacher_resnet50.pth')
        print(f"  → Saved (best: {best_teacher_acc:.2f}%)")

wandb.finish()
print(f"\nTeacher training done. Best: {best_teacher_acc:.2f}%")

[Teacher] Epoch 01/50 | Train: 14.44% | Val: 19.77%
  → Saved (best: 19.77%)
[Teacher] Epoch 02/50 | Train: 24.00% | Val: 30.96%
  → Saved (best: 30.96%)
[Teacher] Epoch 03/50 | Train: 35.36% | Val: 41.95%
  → Saved (best: 41.95%)
[Teacher] Epoch 04/50 | Train: 43.38% | Val: 46.98%
  → Saved (best: 46.98%)
[Teacher] Epoch 05/50 | Train: 50.99% | Val: 52.30%
  → Saved (best: 52.30%)
[Teacher] Epoch 06/50 | Train: 59.07% | Val: 59.75%
  → Saved (best: 59.75%)
[Teacher] Epoch 07/50 | Train: 64.51% | Val: 62.90%
  → Saved (best: 62.90%)
[Teacher] Epoch 08/50 | Train: 69.83% | Val: 65.65%
  → Saved (best: 65.65%)
[Teacher] Epoch 09/50 | Train: 74.16% | Val: 74.06%
  → Saved (best: 74.06%)
[Teacher] Epoch 10/50 | Train: 76.49% | Val: 74.20%
  → Saved (best: 74.20%)
[Teacher] Epoch 11/50 | Train: 78.23% | Val: 69.69%
[Teacher] Epoch 12/50 | Train: 79.05% | Val: 76.69%
  → Saved (best: 76.69%)
[Teacher] Epoch 13/50 | Train: 79.88% | Val: 76.97%
  → Saved (best: 76.97%)
[Teacher] Epoch 14/50 | 

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
lr,███████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
train/acc,▁▂▃▃▄▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
val/acc,▁▂▃▄▄▅▅▆▆▆▆▇▇▇▆▇▇▇▇▇▇▇▇▇▇▇▇█████████████
epoch,50
lr,0.0
train/acc,99.33
val/acc,93.46



Teacher training done. Best: 93.46%


In [18]:
# Load teacher
teacher = torch.load('/content/practicum-project/models/teacher_resnet50.pth',
                     map_location=device, weights_only=False).eval()

# Load student: pruned 50% architecture with fine-tuned weights
def get_cifar_resnet18():
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 10)
    return model

base_model = get_cifar_resnet18().to(device)

example_input = torch.randn(1, 3, 32, 32).to(device)

pruner = tp.pruner.MagnitudePruner(
    base_model,
    example_input,
    importance=tp.importance.MagnitudeImportance(p=1),
    ch_sparsity=0.5,
    ignored_layers=[base_model.fc],
)

pruner.step()

checkpoint = torch.load(
    '/content/practicum-project/models/structured_pruning/pruned/structured_pruned_50pct_fp32.pth',
    map_location=device
)
base_model.load_state_dict(checkpoint['model_state_dict'])
student = base_model.to(device)

def distillation_loss(student_logits, teacher_logits, targets,
                      temperature=4.0, alpha=0.7):
    # Soft loss: KL divergence between softened distributions
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=1),
        F.softmax(teacher_logits  / temperature, dim=1),
        reduction='batchmean'
    ) * (temperature ** 2)

    # Hard loss: standard cross-entropy with true labels
    hard_loss = F.cross_entropy(student_logits, targets)

    return alpha * soft_loss + (1 - alpha) * hard_loss

DISTIL_EPOCHS = 40
TEMPERATURE   = 4.0
ALPHA         = 0.7   # weight on soft loss; 0.7 is a solid default

distil_optimizer = torch.optim.SGD(student.parameters(), lr=1e-3,
                                    momentum=0.9, weight_decay=5e-4)
distil_scheduler = CosineAnnealingLR(distil_optimizer, T_max=DISTIL_EPOCHS, eta_min=1e-6)

wandb.init(
    project="cnn-compression",
    name="distillation-resnet50-to-structured50pct",
    config={
        "epochs": DISTIL_EPOCHS,
        "lr": 1e-3,
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "teacher": "resnet50",
        "student": "resnet18-structured-50pct",
    }
)

In [19]:
best_student_acc = 0.0
for epoch in range(1, DISTIL_EPOCHS + 1):
    student.train()
    teacher.eval()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, targets in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)

        with torch.no_grad():
            teacher_logits = teacher(inputs)

        student_logits = student(inputs)
        loss = distillation_loss(student_logits, teacher_logits, targets,
                                 TEMPERATURE, ALPHA)

        distil_optimizer.zero_grad()
        loss.backward()
        distil_optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        correct    += student_logits.argmax(1).eq(targets).sum().item()
        total      += inputs.size(0)

    distil_scheduler.step()
    val_acc   = evaluate(student, testloader)
    train_acc = 100. * correct / total

    wandb.log({
        "epoch": epoch,
        "train/acc": train_acc,
        "val/acc": val_acc,
        "lr": distil_scheduler.get_last_lr()[0],
    })
    print(f"[Distil] Epoch {epoch:02d}/{DISTIL_EPOCHS} | "
          f"Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")

    if val_acc > best_student_acc:
        best_student_acc = val_acc
        torch.save(student,
            '/content/practicum-project/models/structured_pruning/pruned/structured_pruned_50pct_distilled.pth')
        print(f"  → Saved (best: {best_student_acc:.2f}%)")

wandb.finish()
print(f"\nDistillation done. Best student: {best_student_acc:.2f}%")

[Distil] Epoch 01/40 | Train: 96.58% | Val: 91.99%
  → Saved (best: 91.99%)
[Distil] Epoch 02/40 | Train: 97.46% | Val: 92.46%
  → Saved (best: 92.46%)
[Distil] Epoch 03/40 | Train: 98.02% | Val: 92.86%
  → Saved (best: 92.86%)
[Distil] Epoch 04/40 | Train: 98.39% | Val: 93.23%
  → Saved (best: 93.23%)
[Distil] Epoch 05/40 | Train: 98.67% | Val: 93.45%
  → Saved (best: 93.45%)
[Distil] Epoch 06/40 | Train: 98.84% | Val: 93.43%
[Distil] Epoch 07/40 | Train: 98.93% | Val: 93.48%
  → Saved (best: 93.48%)
[Distil] Epoch 08/40 | Train: 99.08% | Val: 93.56%
  → Saved (best: 93.56%)
[Distil] Epoch 09/40 | Train: 99.13% | Val: 93.70%
  → Saved (best: 93.70%)
[Distil] Epoch 10/40 | Train: 99.26% | Val: 93.66%
[Distil] Epoch 11/40 | Train: 99.27% | Val: 93.77%
  → Saved (best: 93.77%)
[Distil] Epoch 12/40 | Train: 99.38% | Val: 93.82%
  → Saved (best: 93.82%)
[Distil] Epoch 13/40 | Train: 99.40% | Val: 93.79%
[Distil] Epoch 14/40 | Train: 99.45% | Val: 93.65%
[Distil] Epoch 15/40 | Train: 99.51%

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
lr,██████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▃▄▅▆▆▆▆▇▇▇▇▇▇▇▇████████████████████████
val/acc,▁▂▄▅▆▆▆▆▆▆▇▇▇▆▇▆▇▇▇▇▇▇▇▇▇▇███▇███▇█▇▇█▇█
epoch,40
lr,0.0
train/acc,99.754
val/acc,94.11



Distillation done. Best student: 94.22%


In [25]:
import time
student.to("cpu")
student.eval()
dummy = torch.randn(1, 3, 32, 32)

# warmup
for _ in range(20):
    student(dummy)

times = []
for _ in range(100):
    start = time.perf_counter()
    student(dummy)
    times.append((time.perf_counter() - start) * 1000)

print(f"StudentFP32 Latency: {sum(times)/len(times):.2f} ms")

StudentFP32 Latency: 8.62 ms


In [26]:
from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import time
import tracemalloc

device = torch.device('cpu')

model_fp32 = torch.load(
    f'/content/practicum-project/models/structured_pruning/pruned/structured_pruned_50pct_distilled.pth',
    map_location='cpu',
    weights_only=False
).eval()

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
calib_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=False, transform=transform_test)
calib_loader = torch.utils.data.DataLoader(calib_set, batch_size=64,
                                            shuffle=False, num_workers=2)

qconfig = get_default_qconfig('fbgemm')
qconfig_dict = {"": qconfig}

example_input = torch.randn(1, 3, 32, 32)
model_prepared = prepare_fx(model_fp32, qconfig_dict, example_input)

print("Calibrating...")
model_prepared.eval()
with torch.no_grad():
    for i, (inputs, _) in enumerate(calib_loader):
        model_prepared(inputs)
        if i >= 15:  # 16 batches * 64 = ~1024 images, enough
            break

model_int8 = convert_fx(model_prepared)
print("Quantization done.")

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=False, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256,
                                          shuffle=False, num_workers=2)

correct, total = 0, 0
model_int8.eval()
with torch.no_grad():
    for inputs, targets in testloader:
        outputs = model_int8(inputs)
        correct += outputs.argmax(1).eq(targets).sum().item()
        total   += inputs.size(0)
print(f"INT8 Accuracy: {100. * correct / total:.2f}%")

dummy = torch.randn(1, 3, 32, 32)
# warmup
for _ in range(20):
    model_int8(dummy)

times = []
for _ in range(100):
    start = time.perf_counter()
    model_int8(dummy)
    times.append((time.perf_counter() - start) * 1000)
print(f"INT8 Latency: {sum(times)/len(times):.2f} ms")

save_path = '/content/practicum-project/models/structured_pruning/pruned_and_quantized/structured_pruned_50pct_distil_int8.pt'
torch.save(model_int8, save_path)

import os
size_mb = os.path.getsize(save_path) / (1024 ** 2)
print(f"INT8 Size: {size_mb:.2f} MB")

/tmp/ipykernel_3922/2811224843.py:28: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = prepare_fx(model_fp32, qconfig_dict, example_input)


Calibrating...


/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConfigMapping instead.
  prepared = prepare(
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(
/tmp/ipykernel_3922/2811224843.py:38: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e

Quantization done.
INT8 Accuracy: 94.22%
INT8 Latency: 5.28 ms
INT8 Size: 2.76 MB
